<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day22_practice4_%EC%8B%A4%EB%AC%B4%ED%98%95Agent_Function_Calling_SDK_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RAG를 '도구 중 하나'로 품고, 도구를 호출하는 Agent를 LangChain 도구(@tool)와 OpenAI 호환 Function Calling(tools / tool_calls)으로 만든다

In [1]:
!pip install -q langchain langchain-core langchain-openai langchain-chroma langchain-text-splitters langchain-google-genai openai pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 757.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1

In [2]:
import os, requests
import torch
import torch.nn.functional as F

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

API_KEY = _get_api_key()
# API 호출 - openAI 호환 api 형식으로 코드 작성
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai" # Gemini OpenAI 호환

In [4]:
# 셀 1. 신고DB
REPORTS_DB = [
    {"id": 12, "hazard_type": "포트홀", "severity": 4, "location": "강남대로 사거리", "status": "처리중"},
    {"id": 15, "hazard_type": "포트홀", "severity": 4, "location": "서초대로 12",     "status": "접수"},
    {"id": 18, "hazard_type": "포트홀", "severity": 3, "location": "충무로 30",     "status": "완료"},
    {"id": 21, "hazard_type": "균열",   "severity": 2, "location": "을지로 골목",   "status": "접수"},
    {"id": 25, "hazard_type": "맨홀",   "severity": 3, "location": "안국역 3번출구", "status": "처리중"},
]

In [7]:
# 셀 2. 문서 로드 : PDF는 pypdf로 페이지별 텍스트 추출
import glob, os
from langchain_core.documents import Document # langchain은 Document 객체를 입력으로 받기 때문

def load_documents(folder = "/content/drive/MyDrive/KWU/DL/knowledge"):
  docs = []
  for path in sorted(glob.glob(os.path.join(folder, "*"))): # path: 파일 하나의 전체 경로
    name = os.path.basename(path) # 파일명
    if path.endswith(".pdf"):
      from pypdf import PdfReader # 함수/조건문 안에서도 import 가능
      reader = PdfReader(path) # pages 속성에 페이지들이 들어있음
      for i, page in enumerate(reader.pages): # 페이지 번호, 해당 페이지 한 장
        text = page.extract_text() # 글자만 문자열로, 스캔 이미지 PDF면 "" 빈 문자열로
        if text and text.strip(): # A and B 일 때 A가 false면 B는 안봄
          docs.append(Document(page_content=text,metadata={"source":name, "page": i}))
    elif path.endswith((".text", ".md")):
      with open(path, encoding="utf-8") as f:
        text = f.read().strip()
      if text:
        docs.append(Document(page_content=text, metadata={"source":name}))
  return docs

documents = load_documents()
print(f"1. 로드: 문서 {len(documents)}개") # Document 객체 갯수
for d in documents:
  print(f" [{d.metadata["source"]}] {len(d.page_content)}자")

1. 로드: 문서 5개
 [01_도로법_포트홀.pdf] 108자
 [02_맨홀_지침.pdf] 83자
 [03_균열_기준.pdf] 85자
 [04_신고_절차.pdf] 73자
 [05_종합_도로관리_규정.pdf] 632자


In [6]:
# 셀 3. 청킹 : 긴 문서를 검색 단위로 자른다
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"2. 청킹: 문서 {len(documents)}개 → 청크 {len(chunks)}개")

2. 청킹: 문서 5개 → 청크 6개


In [10]:
# 셀 4. 임베딩 + 벡터DB - Gemini 임베딩을 Chroma 벡터 DB에
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 설정 (Gemini)
# OpenAIEmbeddings에 base_url만 Gemini로 바꾸면 Gemini 임베딩이 쓰인다 (OpenAI 호환)
embeddings = OpenAIEmbeddings(
    model = "gemini-embedding-2",
    base_url = GEMINI_BASE_URL,
    api_key = API_KEY,
    check_embedding_ctx_length = False, # OpenAI 전용 토큰검사 끔
)

# 벡터DB 구축 (Chroma)
vectorstore = Chroma.from_documents( # 청크들을 임베딩해서 벡터DB에 적재
    documents = chunks,
    embedding = embeddings,
    collection_name = "road_regulations",
    persist_directory = "/content/drive/MyDrive/KWU/DL/chroma_db", # 디스크에 영속 저장
)
print(f"3. 벡터DB 구축 완료: 청크 {len(chunks)}개 적재 (Gemini 임베딩)")

# retriever - 벡터DB를 '검색기' 객체로, k=3: 상위 3개 청크
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

3. 벡터DB 구축 완료: 청크 6개 적재 (Gemini 임베딩)


In [ ]:
# 실습 2에서 도구 3개를 파이썬 함수로 만들고 설명서는 프롬프트 글로 따로 작성했는데
# @tool데코레이터는 이 둘을 합친 것 : 함수이름 + 타입힌트 + docstring이 곧 도구 설명서(스키마)

from langchain_core.tools import tool
def format_docs(docs):
  return "\n\n".join(d.page_content for d in docs) # 한 덩어리로

# 도구 1 : 규정 검색
@tool(pares_docstring=True, response_format="content_and_artifact") # parse_docstring=True는 docstring의  Args 설명을 인자 설명으로 파싱
def search_regulation(query: str): # response_format = (LLM에게 줄 글, 우리가 챙길 데이터)
  """ 도로 위험 처리 규정 문서를 검색한다. 처리 기한, 절차, 담당 부서 등 '규정'을 묻는 질문에 사용한다. 신고 건수나 특정 신고의 상태에는 사용하지 않는다.

  Args:
      query: 검색할 질문 또는 키워드 (예: '심각도 4 포트홀 처리 기한')
  """
  docs = retriever.invoke(query) # top3: list[Document]
  sources = sorted({d.metadata.get("source", "?") for d in docs})
  return format_docs(docs), sources

# 도구 2 : 신고 조회

# 도구 3 : 상태 조회